# Deliverable 2 - INF201 

Submission deadline: 30th Oct 2026

### Hand-in information:
* This is a group submission, so one person submits on behalf of the pair/group
* Submit the Jupyter notebook (ipynb) <i>and</i> a pdf of the same notebooks
    * Should contain code, results/plots and discussion in markdown cells
* Also submit the output files generated by your code
* Include a link to your GitHub repository and a log of the commits
    * To generate the log, paste this into the terminal after your last commit:<br>
    `git log --graph --pretty=format:"%h - %an, %ar : %s" > git_log.txt`
    * The repository should contain a `requirements.txt` with information on your virtual environment
    * Note: Remember to create a new virtual environment for each deliverable
* Include a discussion of GenAI use, for example a summary of the chat log as you have
done earlier
    * This can be a separate file or be included in the Jupyter notebook
* Make a short (5 mins) video discussing your code and results
    * Note that you both have to contribute to the video
    * Either by recording it together
    * Or recording parts separately and editing it together

## Scenario  - Campus Coffee Shop Sales Pipeline
You have been hired as a Python developer for the Campus Coffee Shop Chain. The chain operates several coffee stands across different campus buildings. This is an international campus, so the stands accept payment in several currencies.

Each cash register saves transaction logs as text files. However, the legacy system has several issues:
* Mixed Character Encodings: Older registers write text files using ISO-8859-1 (Latin-1) encoding, while newer registers write files using UTF-8.
* System Noise: Registers intersperse valid sales lines with printer and network debug log lines.
* Nested Folder Hierarchy: Log files are stored across nested subdirectories organized by year, store, and register ID.
* Data Errors: System glitches record occasional extreme price outliers (e.g., $999.00), and prices in NOK are by error registered using EUR-scale values (e.g., 4.50 NOK instead of 45.00 NOK).

Your task is to set up Python code that discovers all log files, cleans the transaction data, calculates price metrics and generates summary reports.<br>
Note: In this assignment, you write the functions you will use in tasks 1-4, and then put everything together in task 5. You should therefore read through the entire assignment text before you start, to fully understand what you will use each function for.

### Overview of inputs and outputs for this assignment
Inputs (in `data/` folder)
* You generate the input data on you own laptop by running the script `generate_data_for_d2.py`
* `data/store_config.json`: A nested JSON file containing store metadata.
* `data/2026/...`: A folder hierarchy containing subdirectories with .log files in mixed character encodings.

Outputs (files your code should generate)
* `cleaned_transactions.csv`: A CSV table containing all parsed, outlier-cleaned, and currency-corrected transactions.
* `hourly_revenue.json`: A JSON report containing total revenue per hour per item.
* `status_pivot_table.csv`: A CSV file containing a pivot table of total transaction revenue grouped by item across status categories.
* `item_refund_ratios.csv`: A CSV table containing refund ratios for each item type.

## Setup
All imports and shared paths used throughout the notebook are collected here, so the notebook can be run from top to bottom.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

# All input data is generated into the data/ folder by generate_data_for_d2.py
DATA_DIR = Path("data")
LOG_ROOT = DATA_DIR / "2026"
CONFIG_PATH = DATA_DIR / "store_config.json"

## Task 1: File ingestion and regular expressions

1. Write a robust file reader function:
    * Use `pathlib.Path` to handle file paths.
    * The function should be able to handle both `UTF-8` and `ISO-8859-1/Latin-1` encoding. You may assume that these are the only two encodings the files can have. Note that the function should handle the encoding, you may not hard-code which encoding each file has.
    * Return a tuple containing two strings: `(file_text_content, encoding_used)`.
2. Write a log parser function that uses regular expressions.
    Log files contain valid sales lines mixed with system noise.<br>
    Valid line format:<br>
    `[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)`<br>
    Noise lines to ignore:<br>
    `--- PRINTER PAPER LOW WARNING ---`<br>
    `=== NETWORK SYNC OK ===`<br>
    * Construct a regular expression (`re` module) to capture the 6 key fields from valid lines:
        * `timestamp` (string, format: `YYYY-MM-DD HH:MM:SS`)
        * `register_id` (string, e.g., `POS_01`)
        * `item` (string, e.g., `Latte`)
        * `price` (float, e.g., `4.50`)
        * `currency` (string, e.g., `EUR`, `NOK`, `USD`)
        * `status` (string: `SUCCESS`, `REFUND`, or `CANCELLED`)
    * Return a list of Python dictionaries, where each dictionary represents one valid transaction record.

### Task 1.1 – Robust file reader
The function tries the possible encodings in a fixed order: **UTF-8 first, then ISO-8859-1 (Latin-1)**.

The order matters. Every possible byte value (0–255) is a valid Latin-1 character, so decoding with Latin-1 *never* raises an error, even when the file is really UTF-8 (the text would then be silently garbled, e.g. `ø` would become `Ã¸`). UTF-8, on the other hand, has strict rules for how non-ASCII characters are built from bytes, so a Latin-1 file containing characters such as `æ`, `ø` or `å` raises a `UnicodeDecodeError` when read as UTF-8. Trying UTF-8 first and falling back to Latin-1 therefore detects the encoding without hard-coding which file uses which encoding.

In [2]:
ENCODINGS = ("utf-8", "iso-8859-1")


def read_log_file(file_path: str | Path) -> tuple[str, str]:
    """Read a text file that is encoded as either UTF-8 or ISO-8859-1 (Latin-1).

    UTF-8 is tried first. If the bytes are not valid UTF-8, the file is read
    as Latin-1 instead. Latin-1 must come last, since it accepts any byte
    sequence and would therefore never fail.

    Parameters
    ----------
    file_path : str | Path
        Path to the text file.

    Returns
    -------
    tuple[str, str]
        (file_text_content, encoding_used)
    """
    path = Path(file_path)

    for encoding in ENCODINGS:
        try:
            return path.read_text(encoding=encoding), encoding
        except UnicodeDecodeError:
            continue  # Not this encoding, try the next one

    # Not reachable with Latin-1 as fallback, but makes the intent explicit
    raise ValueError(f"Could not decode {path} with any of {ENCODINGS}")

### Task 1.2 – Log parser using regular expressions
The regular expression describes the **whole** valid line from start (`^`) to end (`$`), using named groups so each field can be read out by name:

| Part of pattern | Matches | Field |
|---|---|---|
| `\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]` | `[2026-10-15 08:30:00]` | `timestamp` |
| `\[(?P<register_id>\w+)\]` | `[POS_01]` | `register_id` |
| `item=(?P<item>.+?)` | `item=Latte` (also multi-word names) | `item` |
| `price=(?P<price>\d+(?:\.\d+)?)` | `price=4.50` | `price` |
| `(?P<currency>[A-Z]{3})` | `EUR`, `NOK`, `USD` | `currency` |
| `\((?P<status>...)\)` | `(SUCCESS)`, `(REFUND)` or `(CANCELLED)` – nothing else | `status` |

Matching the full line is important: one of the noise lines in the generated data, `[DEBUG] BARCODE SCANNER REBOOTED`, also starts with `[`, so simply checking how a line starts is not enough. Lines that do not match the full pattern (noise or malformed lines) are skipped. The price is converted to `float`; all other fields are kept as strings.

In [3]:
LOG_LINE_PATTERN = re.compile(
    r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s+"
    r"\[(?P<register_id>\w+)\]\s+"
    r"item=(?P<item>.+?)\s+"
    r"price=(?P<price>\d+(?:\.\d+)?)\s+"
    r"(?P<currency>[A-Z]{3})\s+"
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)\s*$"
)


def parse_log_text(text: str) -> list[dict]:
    """Extract valid transaction records from the text content of a log file.

    Noise lines (printer/network/debug messages) and malformed lines do not
    match the pattern and are ignored.

    Parameters
    ----------
    text : str
        The full text content of a log file.

    Returns
    -------
    list[dict]
        One dictionary per valid transaction, with the keys
        timestamp, register_id, item, price (float), currency and status.
    """
    records = []
    for line in text.splitlines():
        match = LOG_LINE_PATTERN.match(line.strip())
        if match is None:
            continue  # Noise or malformed line
        record = match.groupdict()
        record["price"] = float(record["price"])
        records.append(record)
    return records

### Task 1 – Demonstration on the generated data
Two of the generated files are read. According to the generator, `library_cafe/POS_01` was written as UTF-8 and `library_cafe/POS_02` as ISO-8859-1. (Finding *all* log files is done with the recursive function in Task 2.)

Both files are reported as `utf-8`. This is correct and not a bug: the generated files only contain ASCII characters, and ASCII text is stored as exactly the same bytes in UTF-8 and Latin-1. The Latin-1 file is therefore also valid UTF-8, and decoding it either way gives identical text, which the last printed line confirms. The fallback to Latin-1 only happens when a file contains non-ASCII characters (e.g. `æøå`), which is tested below.

In [4]:
for example_file in [LOG_ROOT / "library_cafe" / "POS_01" / "register_sales.log",
                     LOG_ROOT / "library_cafe" / "POS_02" / "register_sales.log"]:
    content, encoding = read_log_file(example_file)
    records = parse_log_text(content)
    n_lines = len(content.splitlines())
    print(f"{example_file}")
    print(f"  encoding used: {encoding}")
    print(f"  lines in file: {n_lines}, valid transactions: {len(records)}, "
          f"ignored lines: {n_lines - len(records)}")
    print(f"  first record:  {records[0]}")
    print(f"  identical to Latin-1 decoding: "
          f"{content == example_file.read_text(encoding='iso-8859-1')}\n")

data/2026/library_cafe/POS_01/register_sales.log
  encoding used: utf-8
  lines in file: 88, valid transactions: 80, ignored lines: 8
  first record:  {'timestamp': '2026-10-15 07:00:35', 'register_id': 'POS_01', 'item': 'Sandwich', 'price': 7.47, 'currency': 'USD', 'status': 'SUCCESS'}
  identical to Latin-1 decoding: True

data/2026/library_cafe/POS_02/register_sales.log
  encoding used: utf-8
  lines in file: 88, valid transactions: 80, ignored lines: 8
  first record:  {'timestamp': '2026-10-15 07:02:06', 'register_id': 'POS_02', 'item': 'Sandwich', 'price': 7.34, 'currency': 'USD', 'status': 'SUCCESS'}
  identical to Latin-1 decoding: True



### Task 1 – Tests
All generated log files contain only ASCII characters, and ASCII text is identical in UTF-8 and Latin-1. Both encodings can therefore read the generated files without errors, so these files alone cannot show whether the encoding detection works. The tests below therefore also create small temporary files with Norwegian characters (`æøå`) in each encoding, and check that the parser keeps valid lines and rejects noise and malformed lines.

In [5]:
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    sample = "[2026-10-15 08:30:00] [POS_01] item=Kanelbolle price=4.50 NOK (SUCCESS)\nBlåbær æøå ÆØÅ\n"

    # Non-ASCII text in each encoding is detected and decoded correctly
    (tmp / "utf8.log").write_text(sample, encoding="utf-8")
    (tmp / "latin1.log").write_text(sample, encoding="iso-8859-1")
    assert read_log_file(tmp / "utf8.log") == (sample, "utf-8")
    assert read_log_file(tmp / "latin1.log") == (sample, "iso-8859-1")

    # Pure ASCII is valid UTF-8, so it is read as UTF-8
    (tmp / "ascii.log").write_text("=== NETWORK SYNC OK ===\n", encoding="iso-8859-1")
    assert read_log_file(tmp / "ascii.log")[1] == "utf-8"

    # Works with a string path as well as a Path object
    assert read_log_file(str(tmp / "latin1.log"))[1] == "iso-8859-1"

# Example line from the assignment text is parsed into the six fields with correct types
assert parse_log_text("[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)") == [{
    "timestamp": "2026-10-15 08:30:00", "register_id": "POS_01", "item": "Latte",
    "price": 4.5, "currency": "EUR", "status": "SUCCESS",
}]

# Noise lines (from the assignment text and the generator) and malformed lines are ignored
rejected = [
    "--- PRINTER PAPER LOW WARNING ---",
    "--- PRINTER PAPER LOW ---",
    "=== NETWORK SYNC OK ===",
    "[DEBUG] BARCODE SCANNER REBOOTED",
    "[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (PENDING)",  # unknown status
    "[2026-10-15 08:30:00] [POS_01] item=Latte price=abc EUR (SUCCESS)",   # price not a number
    "[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR",            # status missing
    "",
]
assert parse_log_text("\n".join(rejected)) == []

# Windows line endings, surrounding whitespace and multi-word items are handled
parsed = parse_log_text("  [2026-10-15 09:00:00] [POS_02] item=Hot Chocolate price=999.00 USD (REFUND)  \r\n")
assert parsed[0]["item"] == "Hot Chocolate" and parsed[0]["price"] == 999.0 and parsed[0]["status"] == "REFUND"

# On all generated files: every line is either a valid record or one of the known noise lines.
# Only folders are visited, so hidden system files such as macOS' .DS_Store are skipped.
known_noise = {"--- PRINTER PAPER LOW ---", "=== NETWORK SYNC OK ===", "[DEBUG] BARCODE SCANNER REBOOTED"}
store_dirs = sorted(d for d in LOG_ROOT.iterdir() if d.is_dir())
register_dirs = [r for s in store_dirs for r in sorted(s.iterdir()) if r.is_dir()]
assert len(register_dirs) == 9  # 3 stores x 3 registers
for register_dir in register_dirs:
    content, _ = read_log_file(register_dir / "register_sales.log")
    records = parse_log_text(content)
    lines = content.splitlines()
    assert len(records) == 80  # the generator writes exactly 80 sales lines per file
    assert len(records) + sum(line in known_noise for line in lines) == len(lines)
    assert all(r["register_id"] == register_dir.name for r in records)

print("All Task 1 tests passed.")

All Task 1 tests passed.


## Task 2: Recursive utility functions

1. Write a recursive directory search function that finds all log files in a nested directory and returns a list of `pathlib.Path` objects:
    * Convert the input path to a `pathlib.Path` object.
    * Iterate through items using `Path.iterdir()`.
    * If an item is a directory, recursively search inside it.
    * If an item is a file with a matching extension (`.log`), add its Path object to a list.
    * Constraint: Do NOT use `Path.glob()`, `Path.rglob()`, or `os.walk()`. You must implement explicit recursion.
2. Metadata about the different stores can be found in the `data/store_config.json` file. This file contains deeply nested dictionaries and lists (see example below). Write a recursive function that:
    * Traverses any combination of nested dictionaries and lists found in this file.
    * Returns a set (type hint: `set[str]`) of all register id strings where `"active": true`.

Example config.json:
```python
{
  "company": "Campus Coffee",
  "regions": [
    {
      "region_name": "Main Campus",
      "stores": [
        {
          "name": "Library Cafe",
          "registers": [
            {"id": "POS_01", "active": true},
            {"id": "POS_02", "active": false}
          ]
        }
      ]
    }
  ]
}
```

### Task 2.1 – Recursive search for log files
The function looks at every item in a folder with `Path.iterdir()`:

* **Folder** → the function calls itself on that folder (the *recursive case*) and adds the files it returns.
* **File ending in `.log`** → the path is added to the list.
* **Anything else** (e.g. `.txt` files or the hidden `.DS_Store` files macOS creates) → ignored.

The recursion stops by itself when a folder has no subfolders left (the *base case*: the loop then only meets files, or nothing at all). No `glob`, `rglob` or `os.walk` is used.

The items are sorted before they are visited, because `iterdir()` returns them in an arbitrary order that can differ between operating systems. Sorting makes the result, and therefore all later output, the same on every computer.

In [6]:
def find_log_files(directory: str | Path, extension: str = ".log") -> list[Path]:
    """Recursively find all files with the given extension in a folder and its subfolders.

    Parameters
    ----------
    directory : str | Path
        The folder to search in.
    extension : str
        File extension to look for, including the dot (default ".log").

    Returns
    -------
    list[Path]
        Paths to all matching files, in sorted order.
    """
    directory = Path(directory)
    if not directory.is_dir():
        raise NotADirectoryError(f"{directory} is not a directory")

    found_files = []
    for item in sorted(directory.iterdir()):
        if item.is_dir():
            # Recursive case: search the subfolder and add whatever it contains
            found_files.extend(find_log_files(item, extension))
        elif item.is_file() and item.suffix == extension:
            found_files.append(item)
    return found_files

### Task 2.2 – Recursive search for active registers in the config file
The config file is first loaded with `json`, which turns JSON objects into Python dictionaries, JSON arrays into lists and JSON `true` into Python `True`. The function then handles three cases:

* **Dictionary** → if it has `"active": True` and a string `"id"`, the id is added. Then the function calls itself on every value in the dictionary, since a value can itself be a dictionary or a list.
* **List** → the function calls itself on every element.
* **Anything else** (strings, numbers, booleans) → nothing to search, so an empty set is returned. This is the base case that stops the recursion.

Because the function only reacts to *what type* each value is, and never to key names such as `"regions"` or `"stores"`, it works for any combination and depth of nested dictionaries and lists. `is True` is used instead of just checking that the value is truthy, so that values such as the string `"true"` or the number `1` are not mistaken for an active register.

In [7]:
def find_active_registers(data: dict | list) -> set[str]:
    """Recursively collect the ids of all registers marked as active.

    Traverses any combination of nested dictionaries and lists, and returns
    the "id" of every dictionary that has "active": True.

    Parameters
    ----------
    data : dict | list
        The loaded JSON content (or any part of it).

    Returns
    -------
    set[str]
        The ids of all active registers.
    """
    active_ids = set()

    if isinstance(data, dict):
        if data.get("active") is True and isinstance(data.get("id"), str):
            active_ids.add(data["id"])
        for value in data.values():
            active_ids |= find_active_registers(value)

    elif isinstance(data, list):
        for element in data:
            active_ids |= find_active_registers(element)

    # Any other type (str, int, bool, None) contains nothing to search
    return active_ids


def load_store_config(config_path: str | Path) -> dict:
    """Load the store configuration JSON file."""
    return json.loads(Path(config_path).read_text(encoding="utf-8"))

### Task 2 – Demonstration on the generated data

In [8]:
log_files = find_log_files(LOG_ROOT)
print(f"Found {len(log_files)} log files:")
for log_file in log_files:
    _, encoding = read_log_file(log_file)
    print(f"  {log_file}  (read as {encoding})")

store_config = load_store_config(CONFIG_PATH)
active_registers = find_active_registers(store_config)
print(f"\nActive registers in {CONFIG_PATH}: {sorted(active_registers)}")

Found 9 log files:
  data/2026/library_cafe/POS_01/register_sales.log  (read as utf-8)
  data/2026/library_cafe/POS_02/register_sales.log  (read as utf-8)
  data/2026/library_cafe/POS_03/register_sales.log  (read as utf-8)
  data/2026/science_building/POS_01/register_sales.log  (read as utf-8)
  data/2026/science_building/POS_02/register_sales.log  (read as utf-8)
  data/2026/science_building/POS_03/register_sales.log  (read as utf-8)
  data/2026/student_union/POS_01/register_sales.log  (read as utf-8)
  data/2026/student_union/POS_02/register_sales.log  (read as utf-8)
  data/2026/student_union/POS_03/register_sales.log  (read as utf-8)

Active registers in data/store_config.json: ['POS_01', 'POS_02', 'POS_03']


**Observation about the config file.** The config file does not fully match the log files. It lists two stores (*Library Cafe* and *Science Building*) with the registers `POS_01`–`POS_04`, where `POS_04` is inactive. The log folders, however, contain three stores (`library_cafe`, `science_building` and `student_union`), each with `POS_01`–`POS_03`. Since the same register id is used in several stores, a set of id strings cannot tell *which* store a register belongs to. With this data, every register that appears in the logs is in the active set, so filtering on active registers in Task 5 keeps all transactions. The inactive `POS_04` has no log files.

### Task 2 – Tests
The tests check the real data, and also build a temporary folder structure with deeper nesting, empty folders, files with other extensions, a hidden `.DS_Store` file and a *folder* whose name ends in `.log` (which should be searched, not returned). The config function is tested on the example from the assignment text and on an extra, more irregular nesting.

In [9]:
import inspect
import tempfile

# --- find_log_files ---
# The real data: 3 stores x 3 registers, in sorted order, all as Path objects
expected = [LOG_ROOT / store / register / "register_sales.log"
            for store in ["library_cafe", "science_building", "student_union"]
            for register in ["POS_01", "POS_02", "POS_03"]]
assert find_log_files(LOG_ROOT) == expected
assert find_log_files(str(DATA_DIR)) == expected  # works with a string and from a higher level
assert all(isinstance(p, Path) for p in find_log_files(LOG_ROOT))

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    for relative in ["a.log", "x/b.log", "x/y/z/deep.log", "x/notes.txt", "x/old.log.bak",
                     ".DS_Store", "x/y/.DS_Store", "folder.log/inside.log"]:
        file = tmp / relative
        file.parent.mkdir(parents=True, exist_ok=True)
        file.write_text("test")
    (tmp / "empty" / "also_empty").mkdir(parents=True)

    found = [p.relative_to(tmp).as_posix() for p in find_log_files(tmp)]
    assert found == ["a.log", "folder.log/inside.log", "x/b.log", "x/y/z/deep.log"]
    assert find_log_files(tmp / "empty") == []
    assert [p.name for p in find_log_files(tmp, extension=".txt")] == ["notes.txt"]

    try:
        find_log_files(tmp / "a.log")  # a file, not a folder
        raise AssertionError("expected NotADirectoryError")
    except NotADirectoryError:
        pass

# The constraint: no glob, rglob or os.walk in the function
source = inspect.getsource(find_log_files)
assert "glob" not in source and "walk" not in source

# --- find_active_registers ---
assert find_active_registers(store_config) == {"POS_01", "POS_02", "POS_03"}

assignment_example = {
    "company": "Campus Coffee",
    "regions": [{"region_name": "Main Campus", "stores": [{"name": "Library Cafe", "registers": [
        {"id": "POS_01", "active": True},
        {"id": "POS_02", "active": False},
    ]}]}],
}
assert find_active_registers(assignment_example) == {"POS_01"}

irregular = [
    {"id": "R1", "active": True},                                     # list at the top level
    [[{"id": "R2", "active": True}]],                                 # lists inside lists
    {"wrapper": {"more": [{"registers": {"id": "R3", "active": True}}]}},
    {"id": "R4", "active": "true"},                                   # string, not True
    {"id": "R5", "active": 1},                                        # number, not True
    {"id": "R6"},                                                     # no "active" key
    {"id": 7, "active": True},                                        # id not a string
    {"id": "R1", "active": True},                                     # duplicate is stored once
]
assert find_active_registers(irregular) == {"R1", "R2", "R3"}
assert find_active_registers({}) == set() and find_active_registers([]) == set()

print("All Task 2 tests passed.")

All Task 2 tests passed.


## Task 3: Functions for cleaning and analysing data using NumPy
1. Write a function for removing price outliers that:
    * Converts input price data to a 1D NumPy float array.
    * Detects hardware glitch outliers ($999.00$) where a value deviates from the mean by more than 3 standard deviations:$$\lvert x - \mu \rvert > 3\sigma$$
    * Replaces detected outlier values with the median of the valid prices.
    * Returns the cleaned 1D array.
2. Write a function `compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray` that:
    * Computes the element-wise refund ratio using NumPy array operations:
    $$\text{Refund Ratio} = \frac{\text{Refund Count}}{\text{Sales Count} + 10^{-6}}$$
    * The addition of $\epsilon = 10^{-6}$ prevents zero-division errors.

### Task 3.1 – Removing price outliers
The function works in four vectorised NumPy steps, without any Python loops:

1. **Convert** the input (a list, a pandas Series or an array) to a 1D `float64` array with `np.array(..., dtype=float).ravel()`. `np.array` makes a *copy*, so the caller's data is never changed.
2. **Detect** outliers with a boolean mask: `np.abs(prices - mean) > 3 * std`.
3. **Compute the replacement value** as the median of the prices that are *not* outliers. The median is used instead of the mean because it is not pulled upwards by extreme values.
4. **Replace** the outliers using the mask: `cleaned[is_outlier] = median_valid`.

`np.std` uses the population standard deviation (`ddof=0`) by default. With hundreds of transactions the difference from the sample standard deviation (`ddof=1`) is negligible, and on this data both versions flag exactly the same values.

Two edge cases are handled: an empty input returns an empty array (the mean of an empty array is undefined), and if every value is identical the standard deviation is 0, so `> 0` is never true and nothing is replaced.

In [10]:
def remove_price_outliers(prices) -> np.ndarray:
    """Replace extreme price outliers with the median of the valid prices.

    A value x is an outlier if |x - mean| > 3 * std, where mean and std are
    computed from all prices.

    Parameters
    ----------
    prices : array-like
        Price values (list, NumPy array, pandas Series, ...).

    Returns
    -------
    np.ndarray
        A new 1D float array where outliers are replaced by the median of the
        non-outlier prices. The input is not modified.
    """
    cleaned = np.array(prices, dtype=float).ravel()  # copy, so the input is not modified
    if cleaned.size == 0:
        return cleaned

    mean = cleaned.mean()
    std = cleaned.std()
    is_outlier = np.abs(cleaned - mean) > 3 * std

    if is_outlier.any():
        median_valid = np.median(cleaned[~is_outlier])
        cleaned[is_outlier] = median_valid

    return cleaned

### Task 3.2 – Refund ratio
The refund ratio is computed element-wise with NumPy array arithmetic, so one call handles all items at once. Adding $\epsilon = 10^{-6}$ to the denominator means an item with zero sales gives $0 / 10^{-6} = 0$ refunds per sale instead of a division-by-zero error. Since $\epsilon$ is tiny compared to any real sales count, it has no practical effect on the other ratios.

In [11]:
def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Compute the element-wise refund ratio: refund_count / (sales_count + 1e-6).

    Parameters
    ----------
    sales_count : np.ndarray
        Number of sales per item.
    refund_count : np.ndarray
        Number of refunds per item (same shape as sales_count).

    Returns
    -------
    np.ndarray
        The refund ratio for each element, as floats.
    """
    epsilon = 1e-6
    sales_count = np.asarray(sales_count, dtype=float)
    refund_count = np.asarray(refund_count, dtype=float)
    return refund_count / (sales_count + epsilon)

### Task 3 – Demonstration on the generated data
All log files are read and parsed with the functions from Task 1 and 2, and the outlier function is applied to all prices. The refund ratio is shown for the counts per item; in Task 4 the same function is called from a pandas function.

In [12]:
all_records = [record
               for log_file in find_log_files(LOG_ROOT)
               for record in parse_log_text(read_log_file(log_file)[0])]
raw_prices = np.array([record["price"] for record in all_records])
cleaned_prices = remove_price_outliers(raw_prices)

mean, std = raw_prices.mean(), raw_prices.std()
replaced = raw_prices != cleaned_prices
print(f"Number of prices:            {raw_prices.size}")
print(f"Mean and std before:         {mean:.2f} and {std:.2f}")
print(f"Outlier threshold:           |x - {mean:.2f}| > {3 * std:.2f}")
print(f"Values replaced:             {replaced.sum()} (all equal to 999.00: {bool(np.all(raw_prices[replaced] == 999.0))})")
print(f"Replaced with median:        {np.unique(cleaned_prices[replaced])}")
print(f"Price range after cleaning:  {cleaned_prices.min():.2f} – {cleaned_prices.max():.2f}")
print(f"Mean and std after:          {cleaned_prices.mean():.2f} and {cleaned_prices.std():.2f}")

# Refund ratio per item: refunds divided by successful sales
items = np.array([record["item"] for record in all_records])
statuses = np.array([record["status"] for record in all_records])
item_names = np.unique(items)
sales = np.array([np.sum((items == name) & (statuses == "SUCCESS")) for name in item_names])
refunds = np.array([np.sum((items == name) & (statuses == "REFUND")) for name in item_names])
print("\nItem         sales  refunds  refund ratio")
for name, s, r, ratio in zip(item_names, sales, refunds, compute_refund_ratio(sales, refunds)):
    print(f"{name:<12} {s:>5}  {r:>7}  {ratio:>12.3f}")

Number of prices:            720
Mean and std before:         33.36 and 167.38
Outlier threshold:           |x - 33.36| > 502.14
Values replaced:             21 (all equal to 999.00: True)
Replaced with median:        [3.99]
Price range after cleaning:  2.50 – 8.00
Mean and std after:          4.34 and 1.45

Item         sales  refunds  refund ratio
Cappuccino      88       17         0.193
Croissant       80       22         0.275
Espresso        92       21         0.228
Latte           84       15         0.179
Muffin          83       12         0.145
Sandwich        86       14         0.163


**Discussion of the outlier method.**

* *It works well here.* The 999.00 glitches make up about 3 % of the prices, while all real prices are between 2.50 and 8.00. The glitches raise the standard deviation a lot, but they are still far above the 3σ threshold, so all of them are caught and no real price is replaced.
* *The 3σ rule has limits.* The outliers themselves inflate the mean and standard deviation. In a small sample a single extreme value can therefore never exceed 3σ (with $n$ values the largest possible distance is $\frac{n-1}{\sqrt{n}}\sigma$, which is below $3\sigma$ for $n \le 10$). This is shown in the tests below. If glitches were very frequent, the same masking effect could also hide them in larger samples. A method based on the median (e.g. the interquartile range) would be more robust, but the assignment asks for the 3σ rule.
* *The replacement is a rough estimate.* One median is used for all items, so a glitched Sandwich (normally about 7.50) gets the same replacement price as an Espresso. Computing the median per item would be more accurate, but we follow the assignment and use the median of all valid prices.
* *Order of the steps in Task 5.* The cleaning is done **before** the NOK prices are multiplied by 10. In the raw data all currencies are on the same EUR scale, so 999.00 stands clearly apart. After the currency correction, NOK prices of around 45 would increase the spread and make the outliers less distinct.

### Task 3 – Tests

In [13]:
# --- remove_price_outliers ---
# Real data: exactly the 999.00 values are replaced, everything else is unchanged
assert cleaned_prices.shape == raw_prices.shape and cleaned_prices.dtype == np.float64
assert np.array_equal(raw_prices == 999.0, replaced)
assert np.array_equal(cleaned_prices[~replaced], raw_prices[~replaced])
assert np.all(cleaned_prices[replaced] == np.median(raw_prices[raw_prices != 999.0]))
assert cleaned_prices.max() < 10

# Small example: one glitch among 20 normal prices is replaced with the median of the normal prices
normal = [3.0, 4.5, 4.2, 3.5, 3.8, 7.5, 3.1, 4.4, 4.0, 3.6,
          7.2, 4.7, 3.9, 3.3, 4.1, 7.0, 3.7, 4.6, 3.4, 4.3]
result = remove_price_outliers(normal + [999.0])
assert result[-1] == np.median(normal)
assert np.array_equal(result[:-1], normal)

# Different input types give a 1D float array, and the input is not modified
original = np.array(normal + [999.0])
copy_before = original.copy()
remove_price_outliers(original)
assert np.array_equal(original, copy_before)
assert remove_price_outliers(pd.Series(normal + [999.0]))[-1] == np.median(normal)
assert remove_price_outliers([[1, 2], [3, 4]]).shape == (4,)           # 2D input is flattened
assert remove_price_outliers([3, 4, 5]).dtype == np.float64            # ints become floats

# Edge cases: empty input, identical values, no outliers
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("error")   # an empty input must not give a "mean of empty slice" warning
    assert remove_price_outliers([]).size == 0
assert np.array_equal(remove_price_outliers([4.5, 4.5, 4.5]), [4.5, 4.5, 4.5])
assert np.array_equal(remove_price_outliers(normal), normal)

# Known limitation (see discussion): in a very small sample the outlier inflates std and is not detected
assert np.array_equal(remove_price_outliers([4.0, 4.0, 4.0, 999.0]), [4.0, 4.0, 4.0, 999.0])

# --- compute_refund_ratio ---
ratio = compute_refund_ratio(np.array([10, 4, 0]), np.array([2, 1, 0]))
assert isinstance(ratio, np.ndarray) and ratio.dtype == np.float64
np.testing.assert_allclose(ratio, [2 / (10 + 1e-6), 1 / (4 + 1e-6), 0.0])

# Zero sales do not cause a division-by-zero warning or error
with np.errstate(all="raise"):
    assert compute_refund_ratio(np.array([0]), np.array([0]))[0] == 0.0
    np.testing.assert_allclose(compute_refund_ratio(np.array([0]), np.array([3])), [3e6])

# The function is element-wise and works for the real counts
np.testing.assert_allclose(compute_refund_ratio(sales, refunds), refunds / (sales + 1e-6))
assert compute_refund_ratio([5, 5], [1, 2]).shape == (2,)   # lists are accepted as well

print("All Task 3 tests passed.")

All Task 3 tests passed.


## Task 4: Data analysis using Pandas
1. Write a function that:
    * Converts the list of transaction dictionaries read from the log files into a `pandas.DataFrame`.
    * Converts the `timestamp` column to `datetime` and sets it as the DataFrame index.
    * Sorts the DataFrame chronologically by timestamp.
2. Write a function that has the DataFrame you just created as input, groups data by item and resamples transactions in 1-hour windows (1h), calculating total price (sum).
3. Write a function that has the DataFrame as input  and creates a pivot table displaying total transaction revenue grouped by item across status categories (`SUCCESS`, `REFUND`, `CANCELLED`).
4. Write a function that has the DataFrame as input and calculates the refund ratio per item by calling the function you made in task 3.

## Task 5: Create full workflow
Combine the functions you have made into a workflow that does the following:
1. Recursively find all log files and extract active register IDs.
2. Ingest text logs and parse records into a list of dictionaries.
3. Build the initial Pandas DataFrame.
4. Apply NumPy outlier cleaning to remove $999.00$ price glitches.
5. Apply currency correction: multiply price by $10.0$ for all transactions where `currency == "NOK"`.
6. Generate resampled reports, pivot tables and refund ratios.
7. Save all output files listed in the introduction (Scenario: Overview of inputs and outputs for this assignment).